# LASA-RxGuard — training on Kaggle

Handwritten prescription OCR (CRNN vs TrOCR) + look-alike/sound-alike drug-confusion risk flagging.

Kaggle instead of Colab because it needs **no Google Drive** — the Drive storage quota is what
blocked the Colab route, and Kaggle persists outputs to `/kaggle/working` on its own disk.

## Before you run anything — two settings, both in the right-hand panel

1. **Accelerator → GPU T4 x2** (or P100). Cell 1 stops the notebook if this is wrong.
2. **Internet → On.** Required for `pip install`, the GitHub clone, the RxNorm API and the font
   download. This is the single most common reason this notebook fails on Kaggle, and the error
   it produces (a DNS failure deep inside pip) does not obviously say "turn on internet".

Both toggles need a **phone-verified** Kaggle account. If you see no GPU option, that is why.

Session limits: 12 h interactive, 9 h for a committed batch run — far more than the ~2 h needed.

## 1. GPU + internet check

Fails fast and loudly rather than dying 40 minutes later.

In [ ]:
import subprocess, socket, torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Right-hand panel -> Accelerator -> GPU T4 x2, then re-run.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB")

try:
    socket.create_connection(("pypi.org", 443), timeout=8).close()
    print("internet: ON")
except OSError:
    raise SystemExit("No internet. Right-hand panel -> Internet -> On (needs phone verification).")

## 2. Get the code

Cloned into `/kaggle/working` so that everything it writes is persisted as notebook output and stays downloadable after the session ends.

In [ ]:
# Get the code into /kaggle/working, so everything written below is persisted
# as notebook output.
#
# Two supported paths:
#   PUBLIC repo  -> clones directly (nothing else to do).
#   PRIVATE repo -> Kaggle cannot clone it anonymously, so upload the project as
#                   a Kaggle Dataset (Add Input -> New Dataset -> upload a zip of
#                   the repo) and this cell picks it up from /kaggle/input.
import os, glob, shutil, subprocess, pathlib

REPO = "https://github.com/TushKum/lasa-rx-guard.git"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/lasa-rx-guard"

def _looks_like_repo(d):
    return os.path.exists(os.path.join(d, "src", "lasa")) and \
           os.path.exists(os.path.join(d, "requirements.txt"))

if not os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", REPO, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed (private repo or internet off):")
        print("  ", (r.stderr or "").strip()[:200])
        print("falling back to /kaggle/input ...")
        shutil.rmtree(REPO_DIR, ignore_errors=True)

        # Search Kaggle inputs for the project root, at a couple of nesting depths.
        cands = [d for pat in ("/kaggle/input/*", "/kaggle/input/*/*")
                 for d in glob.glob(pat) if os.path.isdir(d) and _looks_like_repo(d)]
        if not cands:
            raise SystemExit(
                "Could not obtain the code.\n"
                "  Option A: make the GitHub repo public, then re-run this cell.\n"
                "  Option B: zip the project, then Add Input -> New Dataset -> upload it.\n"
                f"  Currently visible under /kaggle/input: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'nothing'}"
            )
        # Copy (not symlink) so the notebook can write into the tree.
        shutil.copytree(cands[0], REPO_DIR)
        print("using uploaded dataset:", cands[0])

os.chdir(REPO_DIR)
assert _looks_like_repo("."), f"{REPO_DIR} does not look like the project root"
print("cwd:", pathlib.Path.cwd())
print(sorted(p.name for p in pathlib.Path('.').iterdir() if not p.name.startswith('.')))


## 3. Install

Kaggle ships torch built against its own CUDA runtime, so torch is deliberately **not** reinstalled — replacing it is a reliable way to lose GPU support.

In [ ]:
# Everything except torch/torchvision, which Kaggle already provides.
!grep -vE '^(torch|torchvision)' requirements.txt > /tmp/reqs_kaggle.txt
!pip install -q -r /tmp/reqs_kaggle.txt
!pip install -q -e . --no-deps

import torch, lasa
print("lasa imported OK | torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 4. Build the data

Drug corpus (RxNorm + NLEM India + the ISMP/FDA pair list), 25 OFL handwriting fonts, then the synthetic dataset. All seeded, so this reproduces the local run exactly.

In [ ]:
!python -m lasa.data.drug_names --size 2000
!python -m lasa.data.fonts

In [ ]:
import time
t0 = time.time()
# Kaggle gives 4 vCPUs; more workers than that just thrash.
!python -m lasa.data.synth --n-train 48000 --n-val 3000 --n-test 2000 \
    --tier medium --seed 1337 --workers 4
print(f"dataset built in {(time.time()-t0)/60:.1f} min")

Sanity-check before training on it. The splits must be genuinely disjoint or the generalisation numbers mean nothing.

In [ ]:
import csv
from IPython.display import Image, display

!python -m lasa.data.synth --preview
display(Image('outputs/figures/synth_preview.png'))

rows = list(csv.DictReader(open('data/synth/labels.tsv'), delimiter='\t'))
tr_w = {r['text'] for r in rows if r['split'] == 'train'}
tr_f = {r['font'] for r in rows if r['split'] == 'train'}
print("word overlap train/test_unseen_word:",
      len(tr_w & {r['text'] for r in rows if r['split'] == 'test_unseen_word'}), "(must be 0)")
print("font overlap train/test_unseen_font:",
      len(tr_f & {r['font'] for r in rows if r['split'] == 'test_unseen_font'}), "(must be 0)")

## 5. Train the CRNN

~8.4M parameters, CNN + BiLSTM + CTC, from scratch.

Expect CER near 100% through epoch 0 — CTC sits in an all-blank collapse until alignment
emerges, then error falls sharply. Epoch 1 should be well under 20%. If it is still at 100%
after epoch 2, stop and investigate rather than waiting it out.

In [ ]:
import time
t0 = time.time()
!python -m lasa.train.train_crnn \
    --epochs 15 --batch-size 128 --lr 1e-3 --preload \
    --max-minutes 100 --out outputs/checkpoints/crnn_ctc
print(f"\nCRNN wall clock: {(time.time()-t0)/60:.1f} min (budget 120)")

# Weights-only export: 100 MB -> 34 MB, which is what the demo app ships with.
!python scripts/export_weights.py

## 6. Fine-tune TrOCR

`microsoft/trocr-base-handwritten`, ~334M parameters. This is the arm that could not be trained
locally — an Apple M2 managed ~2 img/s at fp32, which was nowhere near enough to move the model
off its pretrained prior. A T4 with fp16 tensor cores is the whole point of coming here.

On CUDA OOM: drop `--batch-size` to 4 and raise `--grad-accum` to 8 (same effective batch), or
switch to `--model microsoft/trocr-small-handwritten`.

In [ ]:
import time
t0 = time.time()
!python -m lasa.train.train_trocr \
    --model microsoft/trocr-base-handwritten \
    --epochs 3 --batch-size 8 --grad-accum 4 --lr 5e-5 \
    --max-train 20000 --max-eval 500 --workers 2 \
    --max-minutes 110 --out outputs/checkpoints/trocr_ft
print(f"\nTrOCR wall clock: {(time.time()-t0)/60:.1f} min (budget 120)")

## 7. Stage-1 comparison

All four test splits. For the CRNN, greedy / beam / lexicon-constrained appear as **separate rows** — lexicon-constrained decoding is a different task and must never be merged into the free-decoding CER.

In [ ]:
!python -m lasa.evaluation.evaluate
from IPython.display import Markdown, display
display(Markdown(open('outputs/reports/ocr_comparison.md').read()))

## 8. Stage-2 LASA evaluation

Retrieval, pair-flagging precision/recall against the ISMP/FDA ground truth, a negative control, and the signal ablation.

In [ ]:
!python -m lasa.risk.evaluate_lasa --n-retrieval 400 --n-negatives 300 --markdown
from IPython.display import Markdown, display
display(Markdown(open('outputs/reports/lasa_eval.md').read()))

## 9. Collect the results

Anything under `/kaggle/working` is saved as notebook output automatically — no Drive, no
browser download prompt. After the run, use **Save Version → Save & Run All (Commit)**, then
grab the files from the version's **Output** tab.

The tarball below is just a convenience so it is one file rather than a tree.

In [ ]:
import shutil, os
shutil.make_archive('/kaggle/working/lasa_outputs', 'zip', 'outputs')
shutil.copy('docs/results.md', '/kaggle/working/results.md')
shutil.copy('outputs/reports/ocr_comparison.md', '/kaggle/working/ocr_comparison.md')
shutil.copy('outputs/reports/lasa_eval.md', '/kaggle/working/lasa_eval.md')
print("in /kaggle/working:")
for f in sorted(os.listdir('/kaggle/working')):
    p = f'/kaggle/working/{f}'
    if os.path.isfile(p):
        print(f"  {f}  ({os.path.getsize(p)/1e6:.1f} MB)")

---
**Reminder:** unvalidated prototype trained on synthetic handwriting, not a medical device. The
domain gap to real prescriber handwriting is unmeasured and remains the largest single source of
uncertainty in these numbers.